In [317]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports

In [318]:
from datasets import load_dataset

from tokenizers import Tokenizer
from transformers import AutoTokenizer,AutoModel
from sentence_transformers import SentenceTransformer,util
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import pipeline

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as stop_words,TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
import warnings
warnings.filterwarnings('ignore')

# Question 1: Character length of `prompt` and `A` column data combined

In [319]:
train=load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",split="train")

In [320]:
def combine(df):
    df['combined_text']=df['prompt']+" "+df['A']
    return df

In [321]:
updated_train=train.map(combine)
updated_train['combined_text'][0]

"Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time."

In [322]:
print(f"Character length of the combined_text string for the row at index 51: {len(updated_train['combined_text'][51])}")

Character length of the combined_text string for the row at index 51: 614


# Question 2: Exact total vocabulary size hardcoded into `bert-base-uncased` tokenizer

In [323]:
bbu=AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"Total vocabulary size: {bbu.vocab_size}")

Total vocabulary size: 30522


# Question 3: Integer ID asasigned to [SEP] token

In [324]:
print(f"Integer ID asasigned to [SEP] token: {bbu.sep_token_id}")

Integer ID asasigned to [SEP] token: 102


# Question 4: Size of tokenized prompt column

In [325]:
encoded=bbu(train['prompt'][:],padding='max_length',truncation=True,max_length=128,return_tensors='pt')
print('Input_IDs Shape:',encoded.input_ids.shape)

Input_IDs Shape: torch.Size([2000, 128])


# Question 6: Exact shape of the last_hidden_state tensor

In [326]:
model=AutoModel.from_pretrained('bert-base-uncased')
inputs=bbu(train['prompt'][0],return_tensors='pt')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [327]:
with torch.no_grad():
    outputs=model(**inputs)
    
print(f"Exact shape of the last_hidden_state tensor: {outputs.last_hidden_state.shape}")

Exact shape of the last_hidden_state tensor: torch.Size([1, 31, 768])


# Question 7: Sum of the first 5 float values in [CLS] vector

In [328]:
print("Sum of the first 5 float values in [CLS] vector:",sum(outputs.last_hidden_state[0][0][:5]))

Sum of the first 5 float values in [CLS] vector: tensor(-1.2001)


# Question 8

In [329]:
bbu2=AutoTokenizer.from_pretrained('bert-base-uncased',output_attentions=True)
model2=AutoModel.from_pretrained('bert-base-uncased',output_attentions=True)
token=bbu2("Light-ion fusion is a technique.",return_tensors='pt')
with torch.no_grad():
    output=model2(**token)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [330]:
attn=output.attentions[-1][0,0]
tokens=bbu2.convert_ids_to_tokens(token["input_ids"][0])
idx=tokens.index('fusion')
print(f"Attention Weight that CLS Token gives to 'fusion': {attn[0,idx]:0.4f}")

Attention Weight that CLS Token gives to 'fusion': 0.1025


# Question 9: MiniLM-L6-v2

In [331]:
model2=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
prompt_vec=model2.encode(train['prompt'][0])
b_vec=model2.encode(train['B'][0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [332]:
print(f"Cosine Similarity: {float(util.cos_sim(prompt_vec,b_vec)):0.4f}")

Cosine Similarity: 0.7658


# Question 10

## MAP@3 Score of MiniLM

In [333]:
preds=[]
for row in train:
    l=[row[i] for i in train.column_names if i not in ['id','answer']]
    prompt=l[0]
    opt=l[1:]
    prompt_vec=model2.encode(prompt,convert_to_tensor=True)
    option_vecs=model2.encode(opt,convert_to_tensor=True)
    cos_sim=util.cos_sim(prompt_vec,option_vecs)
    labels=["A","B","C","D","E"]
    order=(cos_sim.argsort()).tolist()[0]
    preds.append(labels[order[-1]]+' '+labels[order[-2]]+' '+labels[order[-3]])
preds[:5]

['C D B', 'B C D', 'B A D', 'C D B', 'E D A']

In [334]:
def map_at_3(true,pred):
    scores=[]
    for actual,preds in zip(true,pred):
        score=0.0
        for rank,pred in enumerate(preds.split(' '),start=1):
            if pred==actual:
                score=1.0/rank
                break
        scores.append(score)
    return np.mean(scores)

In [335]:
map3=map_at_3(train['answer'],preds)
print(f"MAP@3 Score: {map3:.4f}")

MAP@3 Score: 0.4231


## The number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions

In [336]:
train_df=train.to_pandas()
vec=TfidfVectorizer(stop_words='english')
l=(
    train_df['prompt'] + ' ' +
    train_df['A'] + ' ' +
    train_df['B'] + ' ' +
    train_df['C'] + ' ' +
    train_df['D'] + ' ' +
    train_df['E']
)
voc=vec.fit_transform(l)
print(f"Total number of feature columns: {voc.shape[1]}")

Total number of feature columns: 2762


In [337]:
m1=vec.transform(train['prompt'])
sims=[]
for i in ['A','B','C','D','E']:
    m2=vec.transform(train[i])
    sim=cosine_similarity(m1, m2)
    sims.append(np.diag(sim))
sims=np.column_stack(sims)

predicted=np.array(['A','B','C','D','E'])[np.argsort(sims)]

In [338]:
preds_tfidf=[]
for i in predicted:
    preds_tfidf.append(i[-3:])
preds[:5]

['C D B', 'B C D', 'B A D', 'C D B', 'E D A']

In [339]:
count=0
for i,j,k in zip(train['answer'],preds,preds_tfidf):
    if i not in k and i in j:
        count+=1
print(f"The number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions: {count}")

The number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions: 613


# Question 11: Zero-Shot-Classification

In [340]:
pipe2=pipeline("zero-shot-classification")
pipe2

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

ZeroShotClassificationPipeline: {'model': 'BartForSequenceClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [341]:
ans=pipe2(train['prompt'][1],candidate_labels=[train[i][1] for i in ["A","B","C"]])
print(f"Probability score given to the top-ranked option: {ans['scores'][0]:.4f}")

Probability score given to the top-ranked option: 0.4575


# Question 12: Zero-Shot-Classification with Independent Sigmoid

In [342]:
ans_multi_level=pipe2(train['prompt'][1],candidate_labels=[train[i][1] for i in ["A","B","C"]],multi_label=True)
print(f"Absolute difference between the sum of the 3 probabilities in Softmax and independent Sigmoids: {abs(sum(ans['scores'])-sum(ans_multi_level['scores'])):0.4f}")

Absolute difference between the sum of the 3 probabilities in Softmax and independent Sigmoids: 0.9995


# Question 13: Text-Generation Encoder-Decoder Model

In [344]:
tokenizer=T5Tokenizer.from_pretrained("google/flan-t5-small")
model=T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

input_text=f"Question: {train[0]['prompt']}. Is the correct answer A: {train[0]['A']} or B: {train[0]['B']}? Answer with just the letter A or B."
input_ids=tokenizer(input_text, return_tensors="pt").input_ids

outputs=model.generate(input_ids,max_new_tokens=5)
print("Exact String Output:",tokenizer.decode(outputs[0],skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Exact String Output: B
